### Getting Started with Langchain and OPEN AI 
Setup LangSmith and LangServe <br>
LangChain: prompt templates, models, and output parsers. <br>
build simple application with langchain  <br>
trace your application with langsmith <br>
server your applicaiton with langserve <br>

In [2]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool

load_dotenv()

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What are you made of?"}]}
)
print(response)
print("\nFinal Answer:", response["messages"][-1].content)

{'messages': [HumanMessage(content='What are you made of?', additional_kwargs={}, response_metadata={}, id='7d24b9f9-8a7f-49b7-b6ae-65ab183291a2'), AIMessage(content='I\'m made of algorithms and data! Specifically, I\'m a language model built on complex neural networks that have been trained on a diverse range of text data. My purpose is to understand and generate human-like text based on the input I receive. While I don\'t have a physical form, my "thoughts" or responses are generated through computations in a virtual environment.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 56, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# You were right! We import create_agent here.
from langchain.agents import create_agent

# Load your OpenAI API key from the .env file
load_dotenv()

# 1. Initialize the Tool
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# 2. Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# 3. Create the Agent
system_message = "You are a helpful assistant."
agent = create_agent(llm, tools=[get_weather], system_prompt=system_message)


# 4. Run the Agent
response = agent.invoke({"messages": [("user", "What is the weather in San Francisco?")]})
print("\nFinal Answer:", response["messages"][-1].content)



Final Answer: The weather in San Francisco is always sunny!


###  RAG

In [4]:
from langchain_community.document_loaders import TextLoader

docs = TextLoader('speech.txt')

print(docs)
print(docs.load())

/var/folders/nf/86b29qg1379fl771lbtf6b3h0000gn/T/ipykernel_82740/451849669.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'speech.txt'}, page_content='1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the steps of the Lincoln Memorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, and Tears" by Winston Churchill (1940)On May 13, 1940, shortly a

In [5]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_paths=("https://www.databricks.com/blog/what-are-large-language-models",))
print(loader)
print(loader.load())

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.databricks.com/blog/what-are-large-language-models', 'title': 'What are Large Language Models (LLM)? | Databricks', 'description': 'Learn what large language models are, how LLMs work, key architectures, and enterprise use cases. Comprehensive guide to LLM technology.', 'language': 'en-US'}, page_content='What are Large Language Models (LLM)? | DatabricksSkip to main contentLoginWhy Databricks DiscoverFor App DevelopersFor ExecutivesFor Startups Lakehouse Architecture Databricks AI ResearchCustomersCustomer StoriesPartnersPartner OverviewExplore the Databricks partner ecosystem Partner SpotlightFeatured partner announcementsPartner ProgramExplore benefits, tiers and how to become a partnerCloud ProvidersDatabricks on AWS, Azure and GCPFind a PartnerDiscover Databricks partners for your needsPartner SolutionsFind custom industry and migration solutionsProduct Databricks PlatformPlatform OverviewA unified platform for data, analytics and AIData 

In [6]:
import bs4
loader = WebBaseLoader(web_paths=("https://www.databricks.com/blog/what-are-large-language-models",), bs_kwargs=dict(parse_only=bs4.SoupStrainer(
    class_=("post-title","post-content","post-header")
)))
print(loader)
print(loader.load())

[Document(metadata={'source': 'https://www.databricks.com/blog/what-are-large-language-models'}, page_content='')]


### Text Splitting the Document

In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the actual documents by adding .load() at the end
loader = TextLoader("speech.txt")
docs = loader.load()

# 2. Set up the text splitter
text_splitters = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# 3. Use split_documents (because docs is a list of Document objects)
final_documents = text_splitters.split_documents(docs)

#print(final_documents)
#print(len(final_documents))
print(len(final_documents[0].page_content))
print(len(final_documents[1].page_content))
print(len(final_documents[2].page_content))



495
498
439


### New Text Splitter


In [18]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# 1. Load the actual documents by adding .load() at the end
loader = TextLoader("speech.txt")
docs = loader.load()

# 2. Set up the text splitter
text_splitters = CharacterTextSplitter(separator="\n",chunk_size=500, chunk_overlap=100)

# 3. Use split_documents (because docs is a list of Document objects)
final_documents = text_splitters.split_documents(docs)

print(final_documents)
print(len(final_documents))
print(len(final_documents[0].page_content))
print(len(final_documents[1].page_content))
print(len(final_documents[2].page_content))



Created a chunk of size 899, which is longer than the specified 500


[Document(metadata={'source': 'speech.txt'}, page_content='1. "I Have a Dream" by Martin Luther King Jr. (1963)Delivered on August 28, 1963, on the \nsteps of the Lincoln'), Document(metadata={'source': 'speech.txt'}, page_content='Memorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, a

In [20]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

text = """Chapter 1: The Beginning

Once upon a time there was a brave knight. He lived in a castle on the hill. Every morning he would ride his horse through the village.

The villagers loved him dearly. They would wave and cheer as he passed by.

Chapter 2: The Journey

One day the knight decided to go on a great adventure. He packed his bags and set off into the unknown forest. The trees were tall and the path was dark.

He walked for many hours until he reached a river. The river was wide and deep."""

# CharacterTextSplitter — only splits on "\n\n"
char_splitter = CharacterTextSplitter(chunk_size=150, chunk_overlap=0, separator="\n\n")
char_chunks = char_splitter.split_text(text)

print("=== CharacterTextSplitter ===")
for i, chunk in enumerate(char_chunks):
    print(f"\nChunk {i} ({len(chunk)} chars):\n{chunk}")

print("\n" + "="*50)

# RecursiveCharacterTextSplitter — tries "\n\n", then "\n", then " "
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)
recursive_chunks = recursive_splitter.split_text(text)

print("\n=== RecursiveCharacterTextSplitter ===")
for i, chunk in enumerate(recursive_chunks):
    print(f"\nChunk {i} ({len(chunk)} chars):\n{chunk}")


Created a chunk of size 153, which is longer than the specified 150


=== CharacterTextSplitter ===

Chunk 0 (24 chars):
Chapter 1: The Beginning

Chunk 1 (135 chars):
Once upon a time there was a brave knight. He lived in a castle on the hill. Every morning he would ride his horse through the village.

Chunk 2 (98 chars):
The villagers loved him dearly. They would wave and cheer as he passed by.

Chapter 2: The Journey

Chunk 3 (153 chars):
One day the knight decided to go on a great adventure. He packed his bags and set off into the unknown forest. The trees were tall and the path was dark.

Chunk 4 (79 chars):
He walked for many hours until he reached a river. The river was wide and deep.


=== RecursiveCharacterTextSplitter ===

Chunk 0 (24 chars):
Chapter 1: The Beginning

Chunk 1 (135 chars):
Once upon a time there was a brave knight. He lived in a castle on the hill. Every morning he would ride his horse through the village.

Chunk 2 (98 chars):
The villagers loved him dearly. They would wave and cheer as he passed by.

Chapter 2: The Journey

Chu

### HTML header Text Splitter

In [2]:
from langchain_text_splitters import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter

# The HTML content
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>Machine Learning</h1>
        <p>Machine learning is a subset of artificial intelligence that enables 
        systems to learn and improve from experience without being explicitly programmed.</p>
        
        <h2>Supervised Learning</h2>
        <p>Supervised learning uses labeled datasets to train algorithms to classify data 
        or predict outcomes accurately.</p>
        
        <h3>Classification</h3>
        <p>Classification is a type of supervised learning where the output is a category, 
        such as spam or not spam.</p>
        
        <h3>Regression</h3>
        <p>Regression predicts a continuous output, such as house prices or temperature.</p>
        
        <h2>Unsupervised Learning</h2>
        <p>Unsupervised learning uses unlabeled data to discover hidden patterns 
        or groupings in data.</p>
        
        <h3>Clustering</h3>
        <p>Clustering groups similar data points together, like customer segmentation.</p>
    </div>
</body>
</html>
"""

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
]

# Step 1: Split by HTML headers
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_splits = html_splitter.split_text(html_string)

# Step 2: Further split into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
final_splits = text_splitter.split_documents(html_splits)

for i, doc in enumerate(final_splits):
    print(f"\nChunk {i} ({len(doc.page_content)} chars): {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")



Chunk 0 (16 chars): Machine Learning
  Metadata: {'Header 1': 'Machine Learning'}

Chunk 1 (68 chars): Machine learning is a subset of artificial intelligence that enables
  Metadata: {'Header 1': 'Machine Learning'}

Chunk 2 (81 chars): systems to learn and improve from experience without being explicitly programmed.
  Metadata: {'Header 1': 'Machine Learning'}

Chunk 3 (19 chars): Supervised Learning
  Metadata: {'Header 1': 'Machine Learning', 'Header 2': 'Supervised Learning'}

Chunk 4 (78 chars): Supervised learning uses labeled datasets to train algorithms to classify data
  Metadata: {'Header 1': 'Machine Learning', 'Header 2': 'Supervised Learning'}

Chunk 5 (48 chars): or predict outcomes accurately.  
Classification
  Metadata: {'Header 1': 'Machine Learning', 'Header 2': 'Supervised Learning'}

Chunk 6 (96 chars): Classification  
Classification is a type of supervised learning where the output is a category,
  Metadata: {'Header 1': 'Machine Learning', 'Header 2': 'Supervi

### Embedding Technique
1 . Open AI
2 . Ollama -> open source
3 . Hugging Face

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv() # load all the enviornment variables

#os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") 

In [14]:
from langchain_openai import OpenAIEmbeddings 
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=1024)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x12cc43b80>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x12dc413c0>, model='text-embedding-3-large', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [15]:
text = "This is a tutorial on OPENAI embedding"
query_result = embeddings.embed_query(text)
query_result

[0.0027103424072265625,
 0.0574951171875,
 -0.019073486328125,
 -0.054931640625,
 0.03302001953125,
 0.00560760498046875,
 0.0229339599609375,
 0.08001708984375,
 -0.024444580078125,
 0.0017223358154296875,
 -0.022796630859375,
 -0.005054473876953125,
 -0.01023101806640625,
 -0.0165557861328125,
 0.0186309814453125,
 0.0289306640625,
 0.03387451171875,
 0.07415771484375,
 -0.0271148681640625,
 -0.036102294921875,
 0.02032470703125,
 0.001758575439453125,
 -0.055511474609375,
 -0.051513671875,
 0.03515625,
 -0.0290069580078125,
 0.0202178955078125,
 0.06463623046875,
 -0.0215911865234375,
 0.0662841796875,
 0.0195770263671875,
 -0.014251708984375,
 0.00537872314453125,
 -0.01019287109375,
 -0.023406982421875,
 0.03662109375,
 0.059814453125,
 0.0565185546875,
 -0.06329345703125,
 0.029205322265625,
 0.050933837890625,
 -0.0006799697875976562,
 -0.00033283233642578125,
 -0.0294952392578125,
 -0.05657958984375,
 -0.036651611328125,
 0.00791168212890625,
 -0.0251312255859375,
 -0.028289794

In [16]:
print(len(query_result))

1024


In [19]:
from langchain_community.vectorstores import Chroma 

db = Chroma.from_documents(final_documents, embeddings)

db

In [20]:
query = "I need to get answer"

retrived_results = db.similarity_search(query)
print(retrived_results)

[Document(metadata={'source': 'speech.txt'}, page_content='Memorial during the March on Washington, this is widely considered the defining speech of the American Civil Rights Movement. King galvanized national support for civil rights, famously painting a vision of a future where his four children would "not be judged by the color of their skin but by the content of their character". You can explore the full text and context on the American Rhetoric database.2. "The Gettysburg Address" by Abraham Lincoln (1863)At just 271 words, Lincoln\'s brief remarks made at the dedication of a Soldiers\' National Cemetery in Pennsylvania redefined the purpose of the Civil War. He called for a "new birth of freedom" and enshrined the iconic concept of a government "of the people, by the people, for the people."3. "Blood, Sweat, and Tears" by Winston Churchill (1940)On May 13, 1940, shortly after becoming Prime Minister, Churchill delivered his first address to'), Document(metadata={'source': 'speech